<a href="https://colab.research.google.com/github/chantakorn-uranukul-5792092/chantakorn-uranukul-5792092-AnalyticsEngineering/blob/main/5_02_practice_weekly_ops_attempt3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 5.02 Analytics Engineering: practice test
## Practice: Weekly operations pipeline

## How this practice works

This is a rehearsal for the in-class test. It is **not marked** and nothing is submitted. It works only if you treat it like the real thing: attempt it first, and only then look at the model answer.

This is set to be the same time given in the real test; the same style of notebook and problem; and as close to the real thing as possible. However, of course, the problem is different; the data is different; and you will do it asychronously (which is also different).

- Set a timer for **55 minutes** and work alone. Stop when it rings, then spend **5 minutes** exporting your notebook (see below). Write your start and stop times in the cell under the timing table.
- Work in **Google Colab** and save the notebook as `.ipynb`.
- You may use *any materials, the module GitHub code and any generative AI tool*.
- There is no end-point as such. The task is longer than 55 minutes, so you are completing part of a larger job. Do as much as you can, and do it properly.
- The code itself is not marked. Marks in the real test go on how the task was executed, the methodology applied, and the interpretation of results. Practise that.
- In text cells or comments, include:
  1. **how the code was generated**: the prompt you sent, or the source you used;
  2. **evidence of checking**: what you compared, counted or inspected, and what it showed;
  3. **your interpretation in your own words**.

**Tips**

- Run small pieces and know what output you expect before you run them.
- Put enough text next to the code that someone else can follow your reasoning.
- Follow sound analytics engineering method: understand the grain/rows that you are building around, clean in stages, check before you move on.
- Quality over quantity. A smaller pipeline that you have checked beats a larger one you have not.
- Stay calm. Not finishing is expected.

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Explore and profile the tables |
| 10 to 20 | Plan: grain, joins, cleaning decisions |
| 20 to 45 | Build as far as you can |
| 45 to 55 | Check, reconcile and write up |
| 55 to 60 | Export: **Runtime, then Restart session and run all**; File, Download, .ipynb; put the file in a zip |

In the real test you upload the zip to the Class Test area on my.wbs. Here, keep it so you can compare your work with the model answer afterwards.

**Started at:** ____  **Stopped at:** ____  **Furthest step I reached (see the brief):** ____

## The data

You have six tables from a fictional food-delivery marketplace, Plateful, which works with restaurants in five UK cities. They cover orders placed from 1 March to 30 June 2026.

| Name in this notebook | What it holds |
| --- | --- |
| `orders` | One record per order: customer, restaurant, when it was placed, delivery fee |
| `order_status_events` | The order's progress through the process, recorded as events |
| `order_lines` | The items inside each order |
| `payments` | Money movements for each order: charges and refunds |
| `restaurants` | Restaurant details: cuisine and city |
| `customers` | Customer details |

All timestamps in all tables are stored in **UTC**.

**How the tables connect**

- `orders` to `order_status_events`, `order_lines` and `payments` on `order_id`
- `orders` to `restaurants` on `restaurant_id`
- `orders` to `customers` on `customer_id`

Check the actual column names in the next cell before you write any joins.

In [1]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/practicesession/plateful-practice.zip"    # link to plateful-practice.zip (the CSV files are inside a folder in the zip)

DATA_DIR = "/content/plateful"

if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/plateful.zip")
    zipfile.ZipFile("/content/plateful.zip").extractall(DATA_DIR)

FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}.")

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))

orders: 14,494 rows, 5 columns
    ['order_id', 'customer_id', 'restaurant_id', 'order_ts', 'delivery_fee']
order_status_events: 56,812 rows, 4 columns
    ['event_id', 'order_id', 'status', 'event_ts']
order_lines: 36,120 rows, 5 columns
    ['order_id', 'line_no', 'item_name', 'quantity', 'unit_price']
payments: 16,423 rows, 5 columns
    ['payment_id', 'order_id', 'payment_type', 'amount', 'paid_at']
restaurants: 67 rows, 5 columns
    ['restaurant_id', 'restaurant_name', 'cuisine', 'city', 'effective_from']
customers: 4,200 rows, 3 columns
    ['customer_id', 'signup_date', 'home_city']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each, using only the tables and keys described above. Run them and change them to see what they do. You do not have to use them in your answer.

In [2]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

# Look at the first few rows of a table
display(q("SELECT * FROM orders LIMIT 5"))

# How many rows are there, and how many different order_id values?
display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT order_id) AS n_distinct_orders FROM orders"))

# Join two tables on a key
display(q("""
    SELECT o.order_id, r.restaurant_id
    FROM orders AS o
    JOIN restaurants AS r ON o.restaurant_id = r.restaurant_id
    LIMIT 5
"""))

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


,n_rows,n_distinct_orders
0,14494,14494


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


In [3]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

orders = tables["orders"]
display(orders.head())        # first few rows
print(orders.shape)           # (number of rows, number of columns)
print(orders.dtypes)          # data type of each column

# The same join as in Example 1, using pandas
joined = tables["orders"].merge(tables["restaurants"], on="restaurant_id", how="inner")
display(joined[["order_id", "restaurant_id"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT customer_id FROM orders")
print(result["customer_id"].nunique(), "different customer_id values in orders")

,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00


(14494, 5)
order_id          object
customer_id       object
restaurant_id     object
order_ts          object
delivery_fee     float64
dtype: object


,order_id,restaurant_id
0,O000001,R052
1,O000002,R027
2,O000003,R044
3,O000004,R049
4,O000005,R009


4073 different customer_id values in orders


## Your brief

**From:** Operations

> We run Plateful, a food-delivery marketplace in five UK cities. We are starting a weekly operations report and need the first layers of a data pipeline behind it: clean tables first, then one reporting table. For each week, show orders, net revenue and average delivery time by cuisine and by city. We also need to know how far we can trust the figures.

**What your final table should look like**

One table with columns for week, cuisine, city, number of orders, net revenue and average delivery time in minutes. State clearly what one row of your table represents.

**Decisions you must make and state**

- Which week does an order belong to?
- Which orders count as orders, and which do not?
- What does "net revenue" mean here, and where does it come from?
- What counts as "delivery time", and for which orders can it be measured?

**A suggested route, in order** (you are not expected to finish)

1. Profile the tables: grain, keys, duplicates, odd values.
2. Build clean tables (a silver layer) with one row per order where that is the grain.
3. Build the one reporting table (gold) from them.
4. Show with checks that the gold table agrees with the source data.
5. If you have time: add cancelled orders and a refund rate to the table, and write automated checks for your clean tables.

## 1. Understand the data

*Look before you build. Write down what you find: what one row of each table represents, how the tables connect, and anything that looks odd or missing.*

I will start by understanding the data that we receive by printing out the fisrt 5 rows from each table and also the total number of row from each table

prompt : printing out the first 5 rows from all the tables and also show the total number of row from each table

In [5]:
from IPython.display import display

for name, df in tables.items():
    print(f"=== Table: {name} ({df.shape[0]:,} rows) ===")
    display(df.head())
    print("\n")

=== Table: orders (14,494 rows) ===


,order_id,customer_id,restaurant_id,order_ts,delivery_fee
0,O000001,C01161,R052,2026-03-01T09:13:23Z,1.49
1,O000002,C00723,R027,2026-03-01T09:15:33Z,2.49
2,O000003,C00781,R044,2026-03-01T09:23:42Z,0.00
3,O000004,C01430,R049,2026-03-01T09:28:14Z,1.49
4,O000005,C00039,R009,2026-03-01T11:32:35Z,0.00




=== Table: order_status_events (56,812 rows) ===


,event_id,order_id,status,event_ts
0,E027809,O007104,accepted,2026-04-29T13:28:57Z
1,E018209,O004658,placed,2026-04-08T19:11:30Z
2,E043792,O011178,accepted,2026-06-02T17:07:47Z
3,E021428,O005468,delivered,2026-04-15T13:43:36Z
4,E037134,O009483,picked_up,2026-05-19T12:18:18Z




=== Table: order_lines (36,120 rows) ===


,order_id,line_no,item_name,quantity,unit_price
0,O000001,1,Ramen,3,12.0
1,O000002,1,Soft drink,1,2.0
2,O000002,2,Side salad,1,3.5
3,O000002,3,Burrito,3,8.0
4,O000003,1,Fries,1,3.0


... (2 outputs left out)



=== Table: restaurants (67 rows) ===


,restaurant_id,restaurant_name,cuisine,city,effective_from
0,R064,Fresh Vegan 64,Vegan,London,2025-01-01
1,R012,Royal Burgers 12,Burgers,Bristol,2025-01-01
2,R028,Grand Burgers 28,Burgers,Birmingham,2025-01-01
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
4,R026,Corner Indian 26,Indian,London,2025-01-01




=== Table: customers (4,200 rows) ===


,customer_id,signup_date,home_city
0,C00001,2024-10-15,London
1,C00002,2025-05-17,Leeds
2,C00003,2026-01-25,Leeds
3,C00004,2024-09-14,London
4,C00005,2024-01-25,Manchester


From the sample of the data shown, I don't see any missing value/duplication/formating/inconsistency yet but I did just print only 5 rows so will still make sure to do all the checking for all the tables

I will start the checking with checking for missing value for all 6 tables hopefully there is none

prompt : checking for any missing value from all the tables we've got. show the result by table and by column and the number of missing value found and also filter out those row with missing value in it

In [6]:
# 1. Check for missing values in each table and show summary
for name, df in tables.items():
    missing_counts = df.isnull().sum()
    total_missing = missing_counts.sum()
    print(f"=== Table: {name} (Total missing values: {total_missing}) ===")
    if total_missing > 0:
        print(missing_counts[missing_counts > 0])
        # Filter and display the rows with missing values
        rows_with_nan = df[df.isnull().any(axis=1)]
        print(f"Rows with missing values ({len(rows_with_nan)} rows):")
        display(rows_with_nan.head())
    else:
        print("No missing values found.")
    print("\n" + "="*40 + "\n")

=== Table: orders (Total missing values: 0) ===
No missing values found.


=== Table: order_status_events (Total missing values: 0) ===
No missing values found.


=== Table: order_lines (Total missing values: 0) ===
No missing values found.


=== Table: payments (Total missing values: 0) ===
No missing values found.


=== Table: restaurants (Total missing values: 0) ===
No missing values found.


=== Table: customers (Total missing values: 0) ===
No missing values found.




okay nice. There is no missing value in any of the table. I will next check for any exact duplication in all of the table and hopefully there is none too

prompt : checking for the exact duplication in all 6 tables. show the result by table and filter out all the rows that you identify as duplicate

In [7]:
# 2. Check for exact duplicate rows in each table and show deduplicated statistics
for name, df in tables.items():
    duplicate_mask = df.duplicated()
    num_duplicates = duplicate_mask.sum()
    print(f"=== Table: {name} ===")
    print(f"Total rows: {len(df):,}")
    print(f"Duplicate rows found: {num_duplicates:,}")

    if num_duplicates > 0:
        # Display the actual duplicate rows
        display(df[duplicate_mask].head())
        # Keep only unique rows (filter out duplicates)
        tables[name] = df.drop_duplicates()
        print(f"Shape after filtering duplicates: {tables[name].shape[0]:,} rows")
    else:
        print("No exact duplicate rows found.")
    print("\n" + "="*40 + "\n")

=== Table: orders ===
Total rows: 14,494
Duplicate rows found: 0
No exact duplicate rows found.


=== Table: order_status_events ===
Total rows: 56,812
Duplicate rows found: 0
No exact duplicate rows found.


=== Table: order_lines ===
Total rows: 36,120
Duplicate rows found: 0
No exact duplicate rows found.


=== Table: payments ===
Total rows: 16,423
Duplicate rows found: 30


,payment_id,order_id,payment_type,amount,paid_at
4111,P013877,O012271,charge,19.49,2026-06-11 21:38:21
4665,P000927,O000815,charge,23.99,2026-03-08 11:53:53
5088,P011592,O010249,charge,10.99,2026-05-25 18:52:29
5205,P013414,O011860,charge,52.49,2026-06-08 11:36:08
5224,P001859,O001631,charge,38.49,2026-03-14 22:43:50


Shape after filtering duplicates: 16,393 rows


=== Table: restaurants ===
Total rows: 67
Duplicate rows found: 0
No exact duplicate rows found.


=== Table: customers ===
Total rows: 4,200
Duplicate rows found: 0
No exact duplicate rows found.




There are 30 duplicate rows in the table payment which indicate that the number of unique value of payment_id which is the primary key for payment table will not be the same as the number of row in the payment table so i want to quickly confirm that

prompt : check the number of rows and also the number of unique value of the the primary key for each table. show the result by table and with number of total row and number of unique primary key

In [8]:
# Define the primary keys we expect for each table
primary_keys = {
    "orders": "order_id",
    "order_status_events": "event_id",
    "order_lines": "[order_id, line_no]",  # expected composite key
    "payments": "payment_id",
    "restaurants": "restaurant_id",
    "customers": "customer_id"
}

for name, df in tables.items():
    total_rows = len(df)
    key_col = primary_keys[name]

    if "[" in key_col:  # Composite key evaluation
        cols = [c.strip() for c in key_col.strip("[]").split(",")]
        unique_keys = len(df.groupby(cols))
    else:
        unique_keys = df[key_col].nunique()

    print(f"=== Table: {name} ===")
    print(f"  Primary Key: {key_col}")
    print(f"  Total Rows: {total_rows:,}")
    print(f"  Unique Key Values: {unique_keys:,}")
    print(f"  Is unique key matching row count? {total_rows == unique_keys}")
    print("-" * 40)

=== Table: orders ===
  Primary Key: order_id
  Total Rows: 14,494
  Unique Key Values: 14,494
  Is unique key matching row count? True
----------------------------------------
=== Table: order_status_events ===
  Primary Key: event_id
  Total Rows: 56,812
  Unique Key Values: 56,812
  Is unique key matching row count? True
----------------------------------------
=== Table: order_lines ===
  Primary Key: [order_id, line_no]
  Total Rows: 36,120
  Unique Key Values: 36,120
  Is unique key matching row count? True
----------------------------------------
=== Table: payments ===
  Primary Key: payment_id
  Total Rows: 16,393
  Unique Key Values: 16,393
  Is unique key matching row count? True
----------------------------------------
=== Table: restaurants ===
  Primary Key: restaurant_id
  Total Rows: 67
  Unique Key Values: 64
  Is unique key matching row count? False
----------------------------------------
=== Table: customers ===
  Primary Key: customer_id
  Total Rows: 4,200
  Uniqu

hmm this is a bit unexpected. So there is exact duplicates in the payment table but the unique value of payment_id match with the total number of row. And we found that the number of unique value of restaurant_id which is the primary key for table restaurant does not match. So let investigate more

prompt : filtering out the duplicate row from table payment and filter out the row from table restaurant where the restaurant_id duplicate

In [9]:
# 1. Investigate the duplicate restaurant_ids
rest_df = tables["restaurants"]
dup_ids = rest_df[rest_df.duplicated(subset=['restaurant_id'], keep=False)]['restaurant_id'].unique()
print(f"Duplicated restaurant IDs: {dup_ids}")
print("\nDuplicate rows in restaurants table:")
display(rest_df[rest_df['restaurant_id'].isin(dup_ids)].sort_values('restaurant_id'))

# 2. Deduplicate restaurants by keeping the record with the latest 'effective_from' date
rest_df_sorted = rest_df.sort_values(by=['restaurant_id', 'effective_from'], ascending=[True, True])
cleaned_restaurants = rest_df_sorted.drop_duplicates(subset=['restaurant_id'], keep='last')
tables["restaurants"] = cleaned_restaurants

# 3. Double check the row counts and unique keys again
print("\n=== Verification after Deduplication ===")
print(f"Restaurants original rows: {len(rest_df)}, Cleaned rows: {len(tables['restaurants'])}, Unique restaurant_ids: {tables['restaurants']['restaurant_id'].nunique()}")
print(f"Payments original rows: 16,423, Cleaned rows: {len(tables['payments'])}, Unique payment_ids: {tables['payments']['payment_id'].nunique()}")

Duplicated restaurant IDs: ['R007' 'R021' 'R040']

Duplicate rows in restaurants table:


,restaurant_id,restaurant_name,cuisine,city,effective_from
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
7,R007,Royal Japanese 7,Japanese,London,2026-04-14
6,R021,Grand Thai 21,Thai,LEEDS,2026-05-06
51,R021,Grand Thai 21,Thai,Manchester,2025-01-01
35,R040,Corner Vegan 40,Vegan,Bristol,2026-05-27
48,R040,Corner Vegan 40,Vegan,London,2025-01-01



=== Verification after Deduplication ===
Restaurants original rows: 67, Cleaned rows: 64, Unique restaurant_ids: 64
Payments original rows: 16,423, Cleaned rows: 16393, Unique payment_ids: 16393


Oh the AI just start cleaning the duplication right away without me prompting it. And also it didn't print out the duplicate row from table payment for me

let me redo this. I will break down the step into smaller tast so it's less confusing and will start by checking the duplication in restaurant_id first and use the original table

prompt : filter out the row from table restaurant from the original data where the restaurant_id duplicate


In [10]:
import pandas as pd

# 1. Reload the original restaurants table to ensure we work with fresh data
raw_restaurants = pd.read_csv('/content/plateful/plateful-practice/restaurants.csv')

# 2. Identify the duplicate restaurant_ids
duplicated_mask = raw_restaurants.duplicated(subset=['restaurant_id'], keep=False)
duplicated_restaurants = raw_restaurants[duplicated_mask].sort_values('restaurant_id')

print("=== Duplicated Records in Original Restaurants Table ===")
display(duplicated_restaurants)

# 3. Deduplicate by keeping the latest record based on 'effective_from'
cleaned_restaurants = raw_restaurants.sort_values('effective_from').drop_duplicates(subset=['restaurant_id'], keep='last')

# 4. Save the cleaned DataFrame back to our tables dictionary
tables['restaurants'] = cleaned_restaurants
con.register('restaurants', cleaned_restaurants)

print("\n=== Verification ===")
print(f"Original rows: {len(raw_restaurants)}")
print(f"Cleaned rows: {len(cleaned_restaurants)}")
print(f"Unique restaurant_ids: {cleaned_restaurants['restaurant_id'].nunique()}")

=== Duplicated Records in Original Restaurants Table ===


,restaurant_id,restaurant_name,cuisine,city,effective_from
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
7,R007,Royal Japanese 7,Japanese,London,2026-04-14
6,R021,Grand Thai 21,Thai,LEEDS,2026-05-06
51,R021,Grand Thai 21,Thai,Manchester,2025-01-01
35,R040,Corner Vegan 40,Vegan,Bristol,2026-05-27
48,R040,Corner Vegan 40,Vegan,London,2025-01-01



=== Verification ===
Original rows: 67
Cleaned rows: 64
Unique restaurant_ids: 64


it kept cleaning it on it's own. let me be more clear this time

prompt : filter out the row from table restaurant from the original data where the restaurant_id duplicate. do not clean it

In [11]:
# Load the original restaurants table
original_restaurants = pd.read_csv('/content/plateful/plateful-practice/restaurants.csv')

# Filter for only the rows where restaurant_id has a duplicate
duplicated_rows = original_restaurants[original_restaurants.duplicated(subset=['restaurant_id'], keep=False)]

# Sort and display the duplicated rows
display(duplicated_rows.sort_values(by='restaurant_id'))

,restaurant_id,restaurant_name,cuisine,city,effective_from
3,R007,Royal Japanese 7,Japanese,Bristol,2025-01-01
7,R007,Royal Japanese 7,Japanese,London,2026-04-14
6,R021,Grand Thai 21,Thai,LEEDS,2026-05-06
51,R021,Grand Thai 21,Thai,Manchester,2025-01-01
35,R040,Corner Vegan 40,Vegan,Bristol,2026-05-27
48,R040,Corner Vegan 40,Vegan,London,2025-01-01


okay finlally. It seems like there are some duplication cause the restaurant have many location so I will just make note of that for now

I will now filtering out the duplicate row from payment and investigate why there are exact duplicate but the number of unique value in payment_id match with the number of total row


prompt : print out the duplication rows from table payemnt. do not clean it

In [12]:
# Load the original payments table
original_payments = pd.read_csv('/content/plateful/plateful-practice/payments.csv')

# Identify exact duplicate rows in the payments dataset
duplicated_payments_mask = original_payments.duplicated(keep=False)
duplicated_payments_rows = original_payments[duplicated_payments_mask]

# Display the duplicates sorted by payment_id to inspect them
print(f"Total duplicated rows found: {len(original_payments[original_payments.duplicated()]):,}")
display(duplicated_payments_rows.sort_values(by='payment_id'))

Total duplicated rows found: 30


,payment_id,order_id,payment_type,amount,paid_at
5949,P000219,O000185,charge,25.49,2026-03-02 19:05:43
13373,P000219,O000185,charge,25.49,2026-03-02 19:05:43
1673,P000927,O000815,charge,23.99,2026-03-08 11:53:53
4665,P000927,O000815,charge,23.99,2026-03-08 11:53:53
6582,P000998,O000874,charge,50.99,2026-03-08 19:19:26
3747,P000998,O000874,charge,50.99,2026-03-08 19:19:26
975,P001859,O001631,charge,38.49,2026-03-14 22:43:50
5224,P001859,O001631,charge,38.49,2026-03-14 22:43:50
15431,P002189,O001926,charge,41.99,2026-03-17 18:22:24
9230,P002189,O001926,charge,41.99,2026-03-17 18:22:24


hmm so I verify that it is exact duplicate and even the payment_id is duplicate which confuse me a bit now how the number of unique value if payment_id still match the total number of row

let me ask AI to understand what could be the cause of this

oh and i notice that AI still pulling data from orignal source from my earlier prompt so I will tell it to stop to reduce unnessesary load

prompt : you don't have to load from original data anymore. but explain what could be the possible reason why the the number of unique value if payment_id still match the total number of row when there is exact duplication

okay so from AI response. AI clean the duplication out without me prompting it that is why orginally the total number of row was 16,423 and when I check the number of unique primary key against the total row again the number reduce it to 16,393 row.

so i will reload the original data for all the table to make sure im using the correct version now and verify again. I expact to see that the number of row and primary key count for payment to be off by 30 and for restaurant to be off by 3

prompt : reload the data and show the number of row and unique primary key by table

In [13]:
import os
import glob
import pandas as pd
import duckdb

# 1. Reload original clean data from the CSV directory
DATA_DIR = "/content/plateful"
FILES = {
    "orders": "orders.csv",
    "order_status_events": "order_status_events.csv",
    "order_lines": "order_lines.csv",
    "payments": "payments.csv",
    "restaurants": "restaurants.csv",
    "customers": "customers.csv",
}

# Reload into the tables dictionary
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    tables[name] = pd.read_csv(found[0])
    # Re-register with DuckDB
    con.register(name, tables[name])

# 2. Define expected primary keys
primary_keys = {
    "orders": "order_id",
    "order_status_events": "event_id",
    "order_lines": "[order_id, line_no]",
    "payments": "payment_id",
    "restaurants": "restaurant_id",
    "customers": "customer_id"
}

# 3. Calculate and print raw row counts and unique primary keys
print("=== Original Uncleaned Tables: Row Counts & Unique Keys ===")
for name, df in tables.items():
    total_rows = len(df)
    key_col = primary_keys[name]

    if "[" in key_col:
        cols = [c.strip() for c in key_col.strip("[]").split(",")]
        unique_keys = len(df.groupby(cols))
    else:
        unique_keys = df[key_col].nunique()

    print(f"Table: {name:<20}")
    print(f"  Primary Key:         {key_col}")
    print(f"  Total Rows:          {total_rows:,}")
    print(f"  Unique Key Values:   {unique_keys:,}")
    print(f"  Is Key Unique?       {total_rows == unique_keys}")
    print("-" * 50)

=== Original Uncleaned Tables: Row Counts & Unique Keys ===
Table: orders              
  Primary Key:         order_id
  Total Rows:          14,494
  Unique Key Values:   14,494
  Is Key Unique?       True
--------------------------------------------------
Table: order_status_events 
  Primary Key:         event_id
  Total Rows:          56,812
  Unique Key Values:   56,812
  Is Key Unique?       True
--------------------------------------------------
Table: order_lines         
  Primary Key:         [order_id, line_no]
  Total Rows:          36,120
  Unique Key Values:   36,120
  Is Key Unique?       True
--------------------------------------------------
Table: payments            
  Primary Key:         payment_id
  Total Rows:          16,423
  Unique Key Values:   16,393
  Is Key Unique?       False
--------------------------------------------------
Table: restaurants         
  Primary Key:         restaurant_id
  Total Rows:          67
  Unique Key Values:   64
  Is Key Uniq

okay nice. now we are on the right track

I will move on to check for the formating of the catagorical data now which are status from order_status_events, item_name from order_lines, payment_type from payments, cuisine and city from restaurant and home_city from consutomer

prompt : filter out the unique value and count of these column : status from order_status_events, item_name from order_lines, payment_type from payments, cuisine and city from restaurant and home_city from consutomer. no need to load original data again

In [14]:
# Target columns to inspect
targets = [
    ("order_status_events", "status"),
    ("order_lines", "item_name"),
    ("payments", "payment_type"),
    ("restaurants", "cuisine"),
    ("restaurants", "city"),
    ("customers", "home_city")
]

for table_name, column_name in targets:
    df = tables[table_name]
    print(f"=== Unique Values and Counts for {table_name}.{column_name} ===")
    counts = df[column_name].value_counts(dropna=False)
    display(counts.to_frame())
    print("\n" + "="*50 + "\n")

=== Unique Values and Counts for order_status_events.status ===


,count
status,
placed,14494
accepted,14154
picked_up,13745
delivered,13549
cancelled,870




=== Unique Values and Counts for order_lines.item_name ===


,count
item_name,
Pad thai,3101
Dessert,3054
Soft drink,3034
Fried rice,3022
Cheeseburger,3016
Burrito,3008
Chicken tikka,3005
Margherita,2994
Fries,2982




=== Unique Values and Counts for payments.payment_type ===


,count
payment_type,
charge,14524
refund,1899


... (2 outputs left out)



=== Unique Values and Counts for restaurants.city ===


,count
city,
London,23
Manchester,17
Birmingham,12
Bristol,7
Leeds,5
LEEDS,1
MANCHESTER,1
LONDON,1




=== Unique Values and Counts for customers.home_city ===


,count
home_city,
London,1541
Manchester,857
Birmingham,697
Bristol,569
Leeds,536


okay there are formating issue only in restaurant table in column cuisince and city where there is some casing issue. I already make a note of that in my plan section

I will next check for the formating error for datetime data which are order_ts from table orders, event_ts from table order_status_events, paid_at from table payment, effective_from from table restaurants and signup_date from customers. hopefully it's all using consistent formating

prompt : check whether datetime formating in order_ts from table orders, event_ts from table order_status_events, paid_at from table payment, effective_from from table restaurants and signup_date from customers are consistent. if it not print out a few sample

In [15]:
datetime_targets = [
    ("orders", "order_ts"),
    ("order_status_events", "event_ts"),
    ("payments", "paid_at"),
    ("restaurants", "effective_from"),
    ("customers", "signup_date")
]

for table_name, column_name in datetime_targets:
    df = tables[table_name]
    print(f"=== Inspecting formats for {table_name}.{column_name} ===")
    print(f"Data Type: {df[column_name].dtype}")
    print("Sample values:")
    display(df[column_name].dropna().head(5).tolist())

    # Attempt standard UTC/ISO datetime conversion with format detection
    converted = pd.to_datetime(df[column_name], errors='coerce')
    nulls_after_conversion = converted.isna().sum()
    print(f"Total values: {len(df):,}")
    print(f"Failed to parse as valid datetimes: {nulls_after_conversion:,}")
    if nulls_after_conversion > 0:
        print("Sample unparsed rows:")
        display(df[df[column_name].isna() | converted.isna()].head(5))
    print("\n" + "="*60 + "\n")

=== Inspecting formats for orders.order_ts ===
Data Type: object
Sample values:


['2026-03-01T09:13:23Z',
 '2026-03-01T09:15:33Z',
 '2026-03-01T09:23:42Z',
 '2026-03-01T09:28:14Z',
 '2026-03-01T11:32:35Z']

Total values: 14,494
Failed to parse as valid datetimes: 0


=== Inspecting formats for order_status_events.event_ts ===
Data Type: object
Sample values:


['2026-04-29T13:28:57Z',
 '2026-04-08T19:11:30Z',
 '2026-06-02T17:07:47Z',
 '2026-04-15T13:43:36Z',
 '2026-05-19T12:18:18Z']

Total values: 56,812
Failed to parse as valid datetimes: 0


=== Inspecting formats for payments.paid_at ===
Data Type: object
Sample values:


['2026-06-18 21:41:37',
 '2026-03-31 18:04:12',
 '2026-05-13 16:26:11',
 '2026-06-21 22:24:53',
 '2026-05-25 11:56:06']

Total values: 16,423
Failed to parse as valid datetimes: 0


=== Inspecting formats for restaurants.effective_from ===
Data Type: object
Sample values:


['2025-01-01', '2025-01-01', '2025-01-01', '2025-01-01', '2025-01-01']

Total values: 67
Failed to parse as valid datetimes: 0


=== Inspecting formats for customers.signup_date ===
Data Type: object
Sample values:


['2024-10-15', '2025-05-17', '2026-01-25', '2024-09-14', '2024-01-25']

Total values: 4,200
Failed to parse as valid datetimes: 0




okay datetime date for all the column look consistent so next will check the numerical data and the statistic data to identify any potential outlier

prompt : check the statistical data for delivery_fee from table order, quantity and unit_price from order_lines, amount from payments

In [16]:
# 1. Gather targeted tables and columns
numerical_cols = [
    ("orders", "delivery_fee"),
    ("order_lines", "quantity"),
    ("order_lines", "unit_price"),
    ("payments", "amount")
]

# 2. Display summary statistics for each column
for table_name, col_name in numerical_cols:
    df = tables[table_name]
    print(f"=== Descriptive Statistics for {table_name}.{col_name} ===")
    stats = df[col_name].describe()
    display(stats.to_frame())

    # Check for negative values
    negatives = (df[col_name] < 0).sum()
    if negatives > 0:
        print(f"Warning: Found {negatives:,} negative values!")
        display(df[df[col_name] < 0].head())
    else:
        print("No negative values found.")
    print("\n" + "="*50 + "\n")

=== Descriptive Statistics for orders.delivery_fee ===


,delivery_fee
count,14494.000000
mean,2.150676
std,1.104824
min,0.000000
25%,1.490000
50%,2.490000
75%,2.990000
max,3.490000


No negative values found.


=== Descriptive Statistics for order_lines.quantity ===


,quantity
count,36120.000000
mean,1.995903
std,0.813985
min,1.000000
25%,1.000000
50%,2.000000
75%,3.000000
max,3.000000


No negative values found.


=== Descriptive Statistics for order_lines.unit_price ===


,unit_price
count,36120.000000
mean,7.211739
std,3.131357
min,2.000000
25%,5.000000
50%,8.000000
75%,10.500000
max,12.000000


No negative values found.


=== Descriptive Statistics for payments.amount ===


,amount
count,16423.000000
mean,30.988618
std,29.080373
min,-127.490000
25%,14.490000
50%,31.490000
75%,49.990000
max,127.490000


,payment_id,order_id,payment_type,amount,paid_at
6,P010253,O009054,refund,-11.99,2026-05-15 22:14:20
8,P005117,O004520,refund,-10.40,2026-04-09 03:55:12
16,P011082,O009795,refund,-4.21,2026-05-23 21:05:57
20,P000177,O000148,refund,-9.49,2026-03-04 03:48:26
26,P001778,O001559,refund,-5.26,2026-03-15 08:03:05


There are 1,899 negatives value in column amount which would make sense if it all from the payment_type refund so let me confirm that. the rest of the column statistic data look good

prompt : show unique value of column payment_type fomr table payment for all the records that amount is negative

In [17]:
# Filter payments for negative amounts and get unique payment types
negative_payments = tables['payments'][tables['payments']['amount'] < 0]
unique_types_for_negatives = negative_payments['payment_type'].value_counts(dropna=False)

print("=== Unique payment_type values for negative amounts ===")
display(unique_types_for_negatives.to_frame())

# Let's also print a small sample of these rows to be sure
print("\nSample of payments with negative amounts:")
display(negative_payments.head(5))

=== Unique payment_type values for negative amounts ===


,count
payment_type,
refund,1899



Sample of payments with negative amounts:


,payment_id,order_id,payment_type,amount,paid_at
6,P010253,O009054,refund,-11.99,2026-05-15 22:14:20
8,P005117,O004520,refund,-10.40,2026-04-09 03:55:12
16,P011082,O009795,refund,-4.21,2026-05-23 21:05:57
20,P000177,O000148,refund,-9.49,2026-03-04 03:48:26
26,P001778,O001559,refund,-5.26,2026-03-15 08:03:05


okay so it's all refund as expacted

Next I will just check the remaining column for the formating in case we miss something while leaving the primary key and foreign key columns for the last check

prompt : show unique value from column line_no from column order_lines and check the count of column restauant_name from column restaurants compare with the count of column restaurant_id

I expact the line_no column to just have the number from 1-10 and for the restaurant_name to have the same count as restaurant_id

In [18]:
# 1. Inspect unique values in order_lines.line_no
line_no_counts = tables['order_lines']['line_no'].value_counts().sort_index()
print("=== Unique Values and Counts for order_lines.line_no ===")
display(line_no_counts.to_frame())

# 2. Compare unique restaurant_name vs unique restaurant_id
n_unique_ids = tables['restaurants']['restaurant_id'].nunique()
n_unique_names = tables['restaurants']['restaurant_name'].nunique()

print("\n=== Restaurants Comparison ===")
print(f"Number of unique restaurant_id:   {n_unique_ids}")
print(f"Number of unique restaurant_name: {n_unique_names}")

if n_unique_ids != n_unique_names:
    print("\nWarning: The number of unique names does not match the unique IDs!")
    # Display names that might share IDs or have minor formatting variations
    dup_names = tables['restaurants'].groupby('restaurant_name')['restaurant_id'].nunique()
    display(dup_names[dup_names > 1].to_frame())
else:
    print("\nConfirmed: The number of unique restaurant names matches the number of unique restaurant IDs.")

=== Unique Values and Counts for order_lines.line_no ===


,count
line_no,
1,14494
2,10859
3,7179
4,3588



=== Restaurants Comparison ===
Number of unique restaurant_id:   64
Number of unique restaurant_name: 64

Confirmed: The number of unique restaurant names matches the number of unique restaurant IDs.


yeah the result is exactly just what i expected

lastly will check all the ID column to ensure that it all consistent across table so that there would be no problem when i try to join later

prompt : check the formating of all the ID column across 6 table whether it's consistent

In [19]:
import re

id_mappings = {
    'orders': ['order_id', 'customer_id', 'restaurant_id'],
    'order_status_events': ['event_id', 'order_id'],
    'order_lines': ['order_id'],
    'payments': ['payment_id', 'order_id'],
    'restaurants': ['restaurant_id'],
    'customers': ['customer_id']
}

print("=== Checking ID Formatting and Consistency ===\n")
for tbl, cols in id_mappings.items():
    df = tables[tbl]
    print(f"Table: {tbl}")
    for col in cols:
        non_null_vals = df[col].dropna().astype(str)
        lengths = non_null_vals.apply(len).unique()

        # Find the patterns (letter prefix followed by digits)
        patterns = non_null_vals.apply(lambda x: re.sub(r'\d', '#', x)).unique()

        print(f"  Column: {col:<15} | Unique Lengths: {lengths} | Patterns found: {patterns}")
    print("-" * 60)

=== Checking ID Formatting and Consistency ===

Table: orders
  Column: order_id        | Unique Lengths: [7] | Patterns found: ['O######']
  Column: customer_id     | Unique Lengths: [6] | Patterns found: ['C#####']
  Column: restaurant_id   | Unique Lengths: [4] | Patterns found: ['R###']
------------------------------------------------------------
Table: order_status_events
  Column: event_id        | Unique Lengths: [7] | Patterns found: ['E######']
  Column: order_id        | Unique Lengths: [7] | Patterns found: ['O######']
------------------------------------------------------------
Table: order_lines
  Column: order_id        | Unique Lengths: [7] | Patterns found: ['O######']
------------------------------------------------------------
Table: payments
  Column: payment_id      | Unique Lengths: [7] | Patterns found: ['P######']
  Column: order_id        | Unique Lengths: [7] | Patterns found: ['O######']
------------------------------------------------------------
Table: resta

nice the format is consistent acorss the table so we are ready for cleaning

## 2. Plan

*Before you write the main code, write your plan in your own words: what one row of your final table is, which tables you will join and in what order, how many rows you expect at each step, and how you will handle the problems you found.*

All the findings
- 30 exact duplication in table payments
- 3 duplication in column restaurant_id from table restaurant
- formating issue in casing for column cuisine and city from table restaurant

so I will start of by cleaning the payment table by droping all the exact duplication since it will cause a problem when we joining data later

prompt : create a new payment table where we remove all the exact duplicate rows

In [20]:
# Keep track of original rows
original_row_count = len(tables['payments'])

# Deduplicate payments by removing exact duplicate rows
cleaned_payments = tables['payments'].drop_duplicates()

# Save the clean dataframe back to our tables dictionary and register in DuckDB
tables['payments'] = cleaned_payments
con.register('payments', cleaned_payments)

# Verification output
new_row_count = len(cleaned_payments)
print(f"=== Deduplicating Payments Table ===")
print(f"Original rows: {original_row_count:,}")
print(f"Cleaned rows:  {new_row_count:,}")
print(f"Rows removed:  {original_row_count - new_row_count} (Expected: 30)")
print(f"Are payment_ids completely unique now? {cleaned_payments['payment_id'].is_unique}")

=== Deduplicating Payments Table ===
Original rows: 16,423
Cleaned rows:  16,393
Rows removed:  30 (Expected: 30)
Are payment_ids completely unique now? True


nice so the new table have exactly 30 rows less from the original table which is what i expacted since there are 30 duplication

next will quickyl clean the restaurant table as well but i will keep the duplication in restaurant_id but will make note that when I join data I have to join based on the city also. I will just fix the formating for cuisine and city

prompt : create a new table for restaurant what fix the formating for cuisine column and city column to start with capital letter and the rest in lowercase with no whitespace in front or behind the word and also the same

In [21]:
# 1. Show unique values before cleaning to compare
print("=== Before Cleaning ===")
print("Unique cities:", tables['restaurants']['city'].unique())
print("Unique cuisines:", tables['restaurants']['cuisine'].unique())

# 2. Clean 'cuisine' and 'city' columns: strip whitespace and convert to Title Case
tables['restaurants']['city'] = tables['restaurants']['city'].astype(str).str.strip().str.title()
tables['restaurants']['cuisine'] = tables['restaurants']['cuisine'].astype(str).str.strip().str.title()

# 3. Register updated DataFrame back in DuckDB
con.register('restaurants', tables['restaurants'])

# 4. Show unique values after cleaning to verify
print("\n=== After Cleaning ===")
print("Unique cities:", tables['restaurants']['city'].unique())
print("Unique cuisines:", tables['restaurants']['cuisine'].unique())

=== Before Cleaning ===
Unique cities: ['London' 'Bristol' 'Birmingham' 'LEEDS' 'Manchester' 'MANCHESTER' 'Leeds'
 'LONDON']
Unique cuisines: ['Vegan' 'Burgers' 'Japanese' 'Indian' 'Thai' 'Italian' 'mexican'
 'Chinese' 'japanese' 'thai' 'Burgers ' 'Mexican' 'Japanese ' 'Indian '
 'Italian ']

=== After Cleaning ===
Unique cities: ['London' 'Bristol' 'Birmingham' 'Leeds' 'Manchester']
Unique cuisines: ['Vegan' 'Burgers' 'Japanese' 'Indian' 'Thai' 'Italian' 'Mexican'
 'Chinese']


## 3. Build

*Build as far as you can in the time. Short, re-runnable cells are better than one long cell.*

Add text here as you go along. Show that you are staying in control of the build!

Do not have time to start the aggregation of the final table and checking and verifying the final table

## 4. Check

*Show how you know your answer is right. What did you compare, count or spot check, and what did the checks show? If a check failed, say what you changed.*

Add text here as you go along. Show that you are staying in control of the build!

Added the check thoughout the steps

## 5. Prompts and AI notes

*Paste each prompt you used, as you sent it. Then note what you accepted, what you changed or rejected, and why. Include at least one example where the AI was wrong or needed correcting, if that happened.*

Added the prompt thoughout the steps

## 6. What I could not verify

*Be honest about what remains unchecked, and what you would do next with more time.*

Do not have time to verify the clean table properly

Do not have time to do the aggregation for the final table

Do not have time to check and verify the final table

## Self-check before you stop

Tick these off in your head or in a text cell. They mirror the three things the real test is marked on.

**How was the code generated?**
- [ ] Every block of code that matters has a prompt or a source associated with it.
- [ ] I can say what each block is meant to do and what output I expected.

**Evidence of checking**
- [ ] For every table I built, I checked the number of rows against what I expected, and I can say why.
- [ ] I checked that keys I treat as unique are unique, and I looked for duplicates.
- [ ] My final table agrees with the source data in at least one total, and I show the comparison.
- [ ] I looked at individual records, not only at totals.
- [ ] I wrote down at least one thing the checks caught, and what I changed (if possible).

**Interpretation in my own words**
- [ ] I state each decision (week, which orders count, net revenue, delivery time) and why I chose it.
- [ ] I say what the figures can and cannot be trusted for.
- [ ] Section 6 lists what I did not verify, honestly.